# Price Prediction Model — Archive Fashion Items

This notebook builds a price prediction model for the top 5 most-traded archive fashion items on Grailed. We compare **Linear Regression** (baseline) vs **XGBoost** using time-series features and rolling price averages.

**Pipeline:**
1. Data loading & outlier removal
2. Feature engineering (time, condition, rolling averages)
3. Model training (80/20 time-series split)
4. Evaluation (MAE / RMSE comparison)
5. Visualization (actual vs predicted)
6. Summary table with price trend signals

---

In [1]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from IPython.display import Markdown, display

import price_model

print("Modules loaded.")

Modules loaded.


## 1. Load & Clean Data

We load `historical_sold.csv`, filter to items with ≥30 records, and remove price outliers that deviate more than 3x from the median. This prevents extreme listings (mispriced or bundled items) from skewing the model.

In [2]:
df_raw = price_model.load_data()
print(f"Raw data: {len(df_raw)} records, {df_raw['keyword'].nunique()} items")
print(f"Date range: {df_raw['sold_date'].min().strftime('%Y-%m-%d')} ~ {df_raw['sold_date'].max().strftime('%Y-%m-%d')}")
print()

# Per-item counts before cleaning
print("Records per item:")
for kw, g in df_raw.groupby("keyword"):
    print(f"  {kw:<35} {len(g):>3} records, median ${g['sold_price'].median():.0f}")

print("\nRemoving outliers (>3x median)...")
df_clean = price_model.remove_outliers(df_raw)
print(f"\nAfter cleanup: {len(df_clean)} records ({len(df_raw) - len(df_clean)} removed)")

Raw data: 205 records, 5 items
Date range: 2025-11-10 ~ 2026-05-09

Records per item:
  Helmut Lang SS99                     30 records, median $175
  Number Nine AW03                     53 records, median $185
  Number Nine AW09                     50 records, median $185
  Prada bowling shirt                  29 records, median $375
  Vetements oversized hoodie           43 records, median $300

Removing outliers (>3x median)...
  Helmut Lang SS99: removed 3 outliers (median=$175)
  Number Nine AW03: removed 4 outliers (median=$185)
  Number Nine AW09: removed 4 outliers (median=$185)
  Vetements oversized hoodie: removed 2 outliers (median=$300)

After cleanup: 192 records (13 removed)


## 2. Feature Engineering

We extract features from each transaction:

| Feature | Source | Rationale |
|---------|--------|-----------|
| `week_of_year` | sold_date | Captures seasonal demand cycles |
| `month` | sold_date | Monthly trends (holiday spikes, etc.) |
| `day_of_week` | sold_date | Weekend vs weekday buying patterns |
| `days_since_start` | sold_date | Linear time trend |
| `condition_code` | condition | New (4) → Worn (1); better condition = higher price |
| `followers` | followers | Listing hype / demand signal |
| `rolling_avg_7/14/30d` | sold_price history | Recent price momentum — the most predictive features |

In [3]:
print("Building features (rolling averages take a moment)...")
df = price_model.build_features(df_clean)

# Show feature sample
display(df[["keyword", "sold_date", "sold_price", "condition_code", "followers",
            "week_of_year", "month", "day_of_week", "days_since_start",
            "rolling_avg_7d", "rolling_avg_14d", "rolling_avg_30d"]].head(10))

print(f"\nFeature matrix shape: {df[price_model.FEATURE_COLS].shape}")
print(f"Features: {price_model.FEATURE_COLS}")

Building features (rolling averages take a moment)...


,keyword,sold_date,sold_price,condition_code,followers,week_of_year,month,day_of_week,days_since_start,rolling_avg_7d,rolling_avg_14d,rolling_avg_30d
0,Helmut Lang SS99,2025-11-10,210,2,48,46,11,0,0,170.0,170.000000,170.00
1,Helmut Lang SS99,2025-11-12,200,3,7,46,11,2,2,210.0,210.000000,210.00
2,Helmut Lang SS99,2025-11-21,100,3,8,47,11,4,11,170.0,205.000000,205.00
3,Helmut Lang SS99,2025-11-24,110,2,77,48,11,0,14,100.0,170.000000,170.00
4,Helmut Lang SS99,2025-11-26,150,2,45,48,11,2,16,105.0,136.666667,155.00
5,Helmut Lang SS99,2025-12-11,165,3,15,50,12,3,31,170.0,170.000000,140.00
6,Helmut Lang SS99,2025-12-14,120,4,86,50,12,6,34,165.0,165.000000,131.25
7,Helmut Lang SS99,2025-12-21,150,2,96,51,12,6,41,120.0,142.500000,129.00
8,Helmut Lang SS99,2025-12-31,300,3,55,1,12,2,51,170.0,150.000000,145.00
9,Helmut Lang SS99,2026-01-03,100,2,47,1,1,5,54,300.0,225.000000,183.75



Feature matrix shape: (192, 9)
Features: ['week_of_year', 'month', 'day_of_week', 'days_since_start', 'condition_code', 'followers', 'rolling_avg_7d', 'rolling_avg_14d', 'rolling_avg_30d']


## 3. Model Training & Evaluation

We use an **80/20 time-series split** (not random split — this respects temporal ordering and prevents future data leakage). Two models are compared:

- **Linear Regression**: Simple baseline. Assumes linear relationship between features and price.
- **XGBoost**: Gradient-boosted trees. Captures non-linear patterns and feature interactions.

Metrics:
- **MAE** (Mean Absolute Error): Average dollar error — interpretable as "the model is off by $X on average"
- **RMSE** (Root Mean Squared Error): Penalizes large errors more heavily

In [4]:
print("Training models per item...\n")
results = price_model.train_and_evaluate(df)

Training models per item...



  Number Nine AW03: LR MAE=$111, XGB MAE=$94 | Predicted=$146, 30d Avg=$211, Trend=Declining
  Number Nine AW09: LR MAE=$108, XGB MAE=$97 | Predicted=$170, 30d Avg=$211, Trend=Declining


  Vetements oversized hoodie: LR MAE=$143, XGB MAE=$125 | Predicted=$363, 30d Avg=$370, Trend=Stable


  Helmut Lang SS99: LR MAE=$298, XGB MAE=$237 | Predicted=$181, 30d Avg=$264, Trend=Declining
  Prada bowling shirt: LR MAE=$152, XGB MAE=$121 | Predicted=$376, 30d Avg=$380, Trend=Stable


## 4. Visualization — Actual vs Predicted

Each chart shows:
- **Blue dots**: Actual sold prices
- **Red line**: XGBoost predicted trend (or Linear Regression if XGBoost unavailable)
- **Gray dashed line**: Linear Regression baseline
- **Green band**: Train/test split point

In [5]:
for r in results:
    kw = r["keyword"]
    dates = pd.to_datetime(r["dates"])
    actuals = r["actuals"]
    lr_pred = r["lr_pred_all"]
    split_idx = r["train_size"]
    split_date = dates[split_idx]

    fig = go.Figure()

    # Actual prices (blue dots)
    fig.add_trace(go.Scatter(
        x=dates, y=actuals, mode="markers",
        name="Actual Sold Price",
        marker=dict(color="#2563eb", size=7, opacity=0.7),
    ))

    # LR baseline (gray dashed)
    fig.add_trace(go.Scatter(
        x=dates, y=lr_pred, mode="lines",
        name=f"Linear Regression (MAE=${r['lr_mae']:.0f})",
        line=dict(color="#9ca3af", width=2, dash="dash"),
    ))

    # XGBoost (red solid)
    if "xgb_pred_all" in r:
        fig.add_trace(go.Scatter(
            x=dates, y=r["xgb_pred_all"], mode="lines",
            name=f"XGBoost (MAE=${r['xgb_mae']:.0f})",
            line=dict(color="#dc2626", width=2),
        ))

    # Train/test split line (use add_shape to avoid plotly vline bug with dates)
    fig.add_shape(
        type="line",
        x0=split_date, x1=split_date, y0=0, y1=1,
        yref="paper", line=dict(color="#059669", width=1.5, dash="dot"),
    )
    fig.add_annotation(
        x=split_date, y=1.05, yref="paper",
        text="Train | Test", showarrow=False,
        font=dict(color="#059669", size=10),
    )

    trend_emoji = {"Rising": "^", "Declining": "v", "Stable": "-"}[r["trend"]]

    fig.update_layout(
        title=f"{kw}  —  Predicted: ${r['predicted_price']:.0f}  {trend_emoji} {r['trend']} ({r['pct_change']:+.1f}%)",
        xaxis_title="Date", yaxis_title="Price (USD)",
        height=380,
        margin=dict(l=0, r=20, t=50, b=20),
        legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1),
    )
    fig.show()

## 5. Model Comparison — MAE Bar Chart

Side-by-side comparison of Linear Regression vs XGBoost MAE per item. Lower is better.

In [6]:
items = [r["keyword"] for r in results]
lr_maes = [r["lr_mae"] for r in results]

fig = go.Figure()
fig.add_trace(go.Bar(
    x=items, y=lr_maes, name="Linear Regression",
    marker_color="#9ca3af", text=[f"${v:.0f}" for v in lr_maes], textposition="outside",
))

if "xgb_mae" in results[0]:
    xgb_maes = [r["xgb_mae"] for r in results]
    fig.add_trace(go.Bar(
        x=items, y=xgb_maes, name="XGBoost",
        marker_color="#dc2626", text=[f"${v:.0f}" for v in xgb_maes], textposition="outside",
    ))

fig.update_layout(
    title="Model Comparison — MAE per Item (lower is better)",
    yaxis_title="MAE ($)",
    barmode="group",
    height=400,
    margin=dict(l=0, r=20, t=40, b=20),
)
fig.show()

## 6. Summary Table

Final output: predicted price, 30-day average, and trend direction for each item.

- **Rising (↑):** Predicted price > 30-day avg by more than 5%
- **Declining (↓):** Predicted price < 30-day avg by more than 5%
- **Stable (→):** Within ±5% of 30-day average

In [7]:
summary = price_model.build_summary_table(results)
display(summary)

# Markdown interpretation
display(Markdown("---\n### Interpretation\n"))
for r in results:
    emoji = {"Rising": "📈", "Declining": "📉", "Stable": "➡️"}[r["trend"]]
    display(Markdown(
        f"- **{r['keyword']}** {emoji} {r['trend']} ({r['pct_change']:+.1f}%) — "
        f"Predicted **${r['predicted_price']:.0f}** vs 30-day avg ${r['avg_30d']:.0f}  "
        f"(Best model MAE: ${min(r['lr_mae'], r.get('xgb_mae', r['lr_mae'])):.0f})"
    ))

,Item,Records,LR MAE ($),LR RMSE ($),XGB MAE ($),XGB RMSE ($),Predicted Price ($),30-Day Avg ($),Trend,Change (%)
0,Number Nine AW03,49,110.5,128.1,94.4,119.6,146.0,211.0,Declining,-30.7
1,Number Nine AW09,46,107.9,124.8,97.3,119.5,170.0,211.0,Declining,-19.4
2,Vetements oversized hoodie,41,143.1,189.9,124.5,201.1,363.0,370.0,Stable,-1.9
3,Helmut Lang SS99,27,298.3,330.1,236.9,254.3,181.0,264.0,Declining,-31.5
4,Prada bowling shirt,29,152.5,194.7,121.2,147.8,376.0,380.0,Stable,-0.9


---
### Interpretation


- **Number Nine AW03** 📉 Declining (-30.7%) — Predicted **$146** vs 30-day avg $211  (Best model MAE: $94)

- **Number Nine AW09** 📉 Declining (-19.4%) — Predicted **$170** vs 30-day avg $211  (Best model MAE: $97)

- **Vetements oversized hoodie** ➡️ Stable (-1.9%) — Predicted **$363** vs 30-day avg $370  (Best model MAE: $125)

- **Helmut Lang SS99** 📉 Declining (-31.5%) — Predicted **$181** vs 30-day avg $264  (Best model MAE: $237)

- **Prada bowling shirt** ➡️ Stable (-0.9%) — Predicted **$376** vs 30-day avg $380  (Best model MAE: $121)